In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from torchsummary import summary


In [2]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:

transform = transforms.Compose([
    transforms.ToTensor()
])

In [4]:

dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)


In [5]:

def add_noise(inputs, noise_factor=0.5):
    noisy = inputs + noise_factor * torch.randn_like(inputs)
    return torch.clamp(noisy, 0., 1.)


In [6]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super(DenoisingAutoencoder, self).__init__()
        # Encoder: 784 -> 128 -> 64
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        # Decoder: 64 -> 128 -> 784
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid() # Sigmoid scales output between 0 and 1
        )

    def forward(self, x):
        x = x.view(-1, 28 * 28) # Flatten image
        x = self.encoder(x)
        x = self.decoder(x)
        x = x.view(-1, 1, 28, 28) # Reshape back to image format
        return x

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DenoisingAutoencoder().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [8]:

summary(model, input_size=(1, 28, 28))

Layer (type:depth-idx)                   Param #
├─Sequential: 1-1                        --
|    └─Linear: 2-1                       100,480
|    └─ReLU: 2-2                         --
|    └─Linear: 2-3                       8,256
|    └─ReLU: 2-4                         --
├─Sequential: 1-2                        --
|    └─Linear: 2-5                       8,320
|    └─ReLU: 2-6                         --
|    └─Linear: 2-7                       101,136
|    └─Sigmoid: 2-8                      --
Total params: 218,192
Trainable params: 218,192
Non-trainable params: 0


Layer (type:depth-idx)                   Param #
├─Sequential: 1-1                        --
|    └─Linear: 2-1                       100,480
|    └─ReLU: 2-2                         --
|    └─Linear: 2-3                       8,256
|    └─ReLU: 2-4                         --
├─Sequential: 1-2                        --
|    └─Linear: 2-5                       8,320
|    └─ReLU: 2-6                         --
|    └─Linear: 2-7                       101,136
|    └─Sigmoid: 2-8                      --
Total params: 218,192
Trainable params: 218,192
Non-trainable params: 0

In [9]:
def train(model, loader, criterion, optimizer, epochs=5):
    train_loss_history = []
    
    for epoch in range(epochs):
        running_loss = 0.0
        model.train()
        
        for images, _ in loader:
            # 1. Move clean images to device
            images = images.to(device)
            
            # 2. Create noisy version
            noisy_images = add_noise(images).to(device)
            
            # 3. Forward pass (Input noisy, compare to clean)
            optimizer.zero_grad()
            outputs = model(noisy_images)
            loss = criterion(outputs, images)
            
            # 4. Backward pass
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        epoch_loss = running_loss / len(loader)
        train_loss_history.append(epoch_loss)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")
        
    return train_loss_history

# Call the function
train_losses = train(model, train_loader, criterion, optimizer, epochs=3)

Epoch [1/3], Loss: 0.0607
Epoch [2/3], Loss: 0.0348
Epoch [3/3], Loss: 0.0288


In [10]:
def visualize_denoising(model, loader, num_images=10):
    model.eval()
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            noisy_images = add_noise(images).to(device)
            outputs = model(noisy_images)
            break

    images = images.cpu().numpy()
    noisy_images = noisy_images.cpu().numpy()
    outputs = outputs.cpu().numpy()

    print("Name:  lakshmen                ")
    print("Register Number:   212224230137               ")
    plt.figure(figsize=(18, 6))
    for i in range(num_images):
       
        ax = plt.subplot(3, num_images, i + 1)
        plt.imshow(images[i].squeeze(), cmap='gray')
        ax.set_title("Original")
        plt.axis("off")

        ax = plt.subplot(3, num_images, i + 1 + num_images)
        plt.imshow(noisy_images[i].squeeze(), cmap='gray')
        ax.set_title("Noisy")
        plt.axis("off")

        ax = plt.subplot(3, num_images, i + 1 + 2 * num_images)
        plt.imshow(outputs[i].squeeze(), cmap='gray')
        ax.set_title("Denoised")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
train(model, train_loader, criterion, optimizer, epochs=4)
visualize_denoising(model, test_loader)

Epoch [1/4], Loss: 0.0261
Epoch [2/4], Loss: 0.0244
Epoch [3/4], Loss: 0.0229
Epoch [4/4], Loss: 0.0218
Name:  lakshmen                
Register Number:   212224230137               
